## Li-Air Overpotential Prediction

This notebook uses the final NGBoost model to predict Li-air overpotentials from screening-space features generated by the O2 and CO2 models.

Workflow:
1. Load the trained NGBoost model.
2. Read all O2/CO2 screening-space prediction files.
3. Generate prediction means, standard deviations, and 95% prediction intervals.
4. Save the results for each current density.


In [1]:
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

In [2]:
# ============================================================
# 1. Path Settings
# ============================================================

MODEL_FILE = "model/ngboost_final_model.pkl"

INPUT_DIR = Path("data")

OUTPUT_DIR = Path("predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. Load the NGBoost model
# ============================================================

ngb_model = joblib.load(MODEL_FILE)

In [4]:
# ============================================================
# 3. Filter all space-related files
# ============================================================

input_files = sorted(
    INPUT_DIR.glob(
        "tabpfn_predict_O2_CO2_final_*.csv"
    )
)

for file in input_files:
    print(file.name)

tabpfn_predict_O2_CO2_final_100.csv
tabpfn_predict_O2_CO2_final_200.csv
tabpfn_predict_O2_CO2_final_300.csv
tabpfn_predict_O2_CO2_final_400.csv
tabpfn_predict_O2_CO2_final_50.csv
tabpfn_predict_O2_CO2_final_500.csv


In [5]:
# ============================================================
# 4. Predict each file and save the prediction mean,
#    standard deviation, and 95% prediction interval
# ============================================================

for input_file in input_files:

    screening_data = pd.read_csv(input_file)

    # "compound" is the material name, not a model feature
    X_predict = (
        screening_data
        .drop(columns=["compound"])
        .apply(pd.to_numeric, errors="raise")
    )

    if X_predict.isna().any().any():
        missing_columns = X_predict.columns[
            X_predict.isna().any()
        ].tolist()

        raise ValueError(
            f"{input_file.name} contains missing values "
            f"in the following columns: {missing_columns}"
        )

    # Obtain the predictive distribution from NGBoost
    pred_dist = ngb_model.pred_dist(X_predict)

    # Prediction mean of the Normal distribution
    pred_mean = np.asarray(
        pred_dist.params["loc"]
    ).reshape(-1)

    # Prediction standard deviation of the Normal distribution
    pred_std = np.asarray(
        pred_dist.params["scale"]
    ).reshape(-1)

    # Retain all columns from the original dataset
    result = screening_data.copy()

    # Save the prediction mean
    result["overpotential_pred"] = pred_mean

    # Save the prediction standard deviation
    result["overpotential_std"] = pred_std

    # Save the 95% prediction interval
    result["overpotential_lower_95"] = (
        pred_mean - 1.96 * pred_std
    )

    result["overpotential_upper_95"] = (
        pred_mean + 1.96 * pred_std
    )

    # Extract the current density from the filename,
    # such as 50, 100, or 200
    current_density = input_file.stem.split("_")[-1]

    output_file = (
        OUTPUT_DIR
        / f"ngboost_predict_O2_CO2_final_{current_density}.csv"
    )

    result.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"{input_file.name}: "
        f"predicted {len(result)} samples, "
        f"mean standard deviation = {pred_std.mean():.6f}, "
        f"saved as {output_file.name}"
    )

print("\nPrediction of the entire screening space is complete.")

tabpfn_predict_O2_CO2_final_100.csv: predicted 25012 samples, mean standard deviation = 0.080766, saved as ngboost_predict_O2_CO2_final_100.csv
tabpfn_predict_O2_CO2_final_200.csv: predicted 25012 samples, mean standard deviation = 0.077329, saved as ngboost_predict_O2_CO2_final_200.csv
tabpfn_predict_O2_CO2_final_300.csv: predicted 25012 samples, mean standard deviation = 0.080085, saved as ngboost_predict_O2_CO2_final_300.csv
tabpfn_predict_O2_CO2_final_400.csv: predicted 25012 samples, mean standard deviation = 0.077793, saved as ngboost_predict_O2_CO2_final_400.csv
tabpfn_predict_O2_CO2_final_50.csv: predicted 25012 samples, mean standard deviation = 0.089120, saved as ngboost_predict_O2_CO2_final_50.csv
tabpfn_predict_O2_CO2_final_500.csv: predicted 25012 samples, mean standard deviation = 0.101490, saved as ngboost_predict_O2_CO2_final_500.csv

Prediction of the entire screening space is complete.
